# 02 — Clean & Quality Check

> **AI-Assisted Development** — This project was built with [Kiro](https://kiro.dev). See `SOURCES.md` for full attribution.

All cleaning operations are SQL-driven inside DuckDB. This notebook documents
what was done and lets you inspect the results.

### Cleaning operations performed:
1. Renamed FARS `alcohol_fatalities` → `impaired_fatalities_any` (includes drugs/meds, not just alcohol)
2. Verified state name consistency across all 14 source tables
3. Parsed NCSL criminal status text → `first_offense_felony` flag + `felony_threshold`
4. Extracted FARS license status (suspended/revoked drivers in fatal crashes)
5. Standardized NASID enforcement text → boolean/categorical columns (21 fields)
6. Validated BAC testing rates and mandatory testing laws (cross-checked law vs practice)
7. Ran quality checks on all tables
8. Parsed DUI penalty strings → numeric columns (jail days, suspension days, fine amounts)

### Key decisions:
- **FARS 2021+ vs 2015–2020:** Not directly comparable due to schema change. Use `impairment_method` column to filter.
- **For published alcohol-specific analysis:** Use `nhtsa_imputed_2024` (BAC≥.08 only, statistically modeled).
- **FARS `drimpair=9`** captures alcohol + drugs + medication — broader than alcohol alone.
- **felony_threshold = NULL** means DUI is always a misdemeanor in that state (CA, DC, NJ, etc.)

In [ ]:
import sys
sys.path.insert(0, "..")

import duckdb
import pandas as pd
from src.ingest import load_config

cfg = load_config("../config.yaml")
con = duckdb.connect(str("../" + cfg['settings']['duckdb_file']))
print(f"Tables: {len(con.execute('SHOW TABLES').df())}")
con.execute('SHOW TABLES').df()

---
## 1. FARS trends — corrected column names
The raw FARS `drimpair=9` code captures **any impairment** (alcohol, drugs, medication).
Columns renamed from `alcohol_*` to `impaired_*_any` to avoid overstating alcohol's role.

An `impairment_method` flag distinguishes the 2015–2020 methodology (`drunk_dr` field in accident.csv)
from 2021+ (`drimpair` code 9 in a separate file). These are **not directly comparable**.

In [ ]:
# National totals by year — note the methodology break at 2021
national = con.execute("""
    SELECT year, impairment_method,
           SUM(total_fatalities) AS total_fatalities,
           SUM(impaired_fatalities_any) AS impaired_fatalities,
           ROUND(SUM(impaired_fatalities_any) * 100.0 / SUM(total_fatalities), 1) AS pct_impaired
    FROM fars_trends_clean
    GROUP BY year, impairment_method
    ORDER BY year
""").df()
national

---
## 2. State name consistency
All tables use canonical Census Bureau state names (50 states + DC).
NCSL territories (American Samoa, Guam, Puerto Rico, USVI) were filtered out.

In [ ]:
# Verify: all key tables have exactly 51 rows
for table in ['nhtsa_imputed_2024', 'alcohol_consumption', 'dui_arrests_2023',
              'dui_enforcement', 'dui_criminal_status_clean', 'fars_license_status_2024']:
    n = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'  {table}: {n} rows {"✓" if n == 51 else "⚠"}')

---
## 3. Criminal status — felony flags
Parsed from NCSL free-text `criminal_status` column.

- `first_offense_felony`: 1 if any first DUI offense *can* be charged as felony
- `felony_threshold`: which offense number triggers felony (NULL = always misdemeanor)

In [ ]:
felony = con.execute("""
    SELECT state_name, first_offense_felony, felony_threshold, criminal_status
    FROM dui_criminal_status_clean
    ORDER BY felony_threshold, state_name
""").df()
print(f"First offense felony: {felony['first_offense_felony'].sum():.0f} states")
print(f"Threshold distribution: {felony['felony_threshold'].value_counts().sort_index().to_dict()}")
print(f"Always misdemeanor (NULL threshold): {felony['felony_threshold'].isna().sum()} states")
felony

---
## 4. FARS license status — suspended drivers in fatal crashes
From `vehicle.csv` field `L_STATUS`: 1=Suspended, 2=Revoked.

Cross-referenced with `drimpair.csv` to find how many suspended-license drivers
were also impaired at the time of the fatal crash.

**Key finding:** 3,095 drivers in 2024 fatal crashes had suspended/revoked licenses.
22.6% of those were also impaired — suggesting suspension alone doesn't prevent impaired driving.

In [ ]:
lic = con.execute("""
    SELECT state_name, total_drivers_in_fatal_crashes,
           drivers_suspended_revoked, pct_suspended,
           drivers_suspended_and_impaired, pct_suspended_who_impaired
    FROM fars_license_status_2024
    ORDER BY pct_suspended DESC
""").df()
print(f"National: {lic['drivers_suspended_revoked'].sum():,} suspended/revoked in fatal crashes")
print(f"Of those also impaired: {lic['drivers_suspended_and_impaired'].sum():,} ({lic['drivers_suspended_and_impaired'].sum()/lic['drivers_suspended_revoked'].sum()*100:.1f}%)")
lic

---
## 5. NASID enforcement — standardized from free text

**Script:** `scripts/clean_enforcement.py`  
**Input:** `nasid_enforcement` (raw scraped text from NASID state pages)  
**Output:** `nasid_enforcement_clean` (21 columns: booleans + categoricals)

### Transformations applied:
| Raw field | Cleaned column(s) | Method |
|-----------|-------------------|--------|
| `sobriety_checkpoints` | `checkpoints` (cat), `checkpoints_permitted` (bool) | Text match: "Permitted"→1 |
| `no_refusal_programs` | `no_refusal_status` (cat), `no_refusal_active` (bool) | "Utilizes"→active, "Has legal authority"→authorized, "Lacks"→not_authorized |
| `roadside_preliminary_breath_test_pbt_laws` | `pbt_law` (cat), `pbt_authorized` (bool) | "Statute permits"→1 |
| `ignition_interlocks` | `iid_mandate` (cat), `iid_all_offender` (bool) | "Mandatory all offender"→1 |
| `felony_dui` | `felony_dui_threshold` (int), `has_felony_dui` (bool) | "Third offense"→3, "No felony"→NULL |
| `dui_look_back_periods` | `lookback_years` (int) | "Ten years"→10, "Lifetime"→99 |
| `enhanced_penalties_for_high_bac` | `high_bac_threshold` (float), `has_high_bac_penalty` (bool) | "0.15"→0.15, "No enhanced"→NULL |
| `open_container_alcohol` | `open_container` (cat), `open_container_compliant` (bool) | "In compliance"→1 |
| `duid_implied_consent_testing_methods` | `testing_methods` (str), `allows_oral_fluid` (bool) | Contains "Oral"→1 |
| `administrative_license_suspension_revocation` | `als_alr_enacted` (bool) | Contains "enacted"→1 |

### Quality validation:
- All 51 states present
- No unexpected nulls in boolean fields (threshold fields are intentionally NULL for states without that law)
**Post-hoc corrections (2026-08-12):**
- NC and AR: NASID reported "Third offense" but state statutes confirm felony on 4th
  (NC G.S. 20-138.5: 3+ prior convictions; Ark. Code 5-65-111: 4th within 10 years)
- Corrected `felony_dui_threshold` from 3→4 for state_fips 05 (AR) and 37 (NC)


In [ ]:
nasid_clean = con.execute("SELECT * FROM nasid_enforcement_clean ORDER BY state_fips").df()
print(f"{len(nasid_clean)} rows × {len(nasid_clean.columns)} cols")
print()
print("Distribution of key fields:")
print(f"  Checkpoints permitted: {nasid_clean['checkpoints_permitted'].sum()}/51")
print(f"  No-refusal active:     {nasid_clean['no_refusal_active'].sum()}/51")
print(f"  PBT authorized:        {nasid_clean['pbt_authorized'].sum()}/51")
print(f"  IID all-offender:      {nasid_clean['iid_all_offender'].sum()}/51")
print(f"  Has felony DUI:        {nasid_clean['has_felony_dui'].sum()}/51")
print(f"  Open container OK:     {nasid_clean['open_container_compliant'].sum()}/51")
print(f"  Allows oral fluid:     {nasid_clean['allows_oral_fluid'].sum()}/51")
nasid_clean[['state_abbr','checkpoints_permitted','no_refusal_active','pbt_authorized','iid_all_offender','lookback_years','high_bac_threshold']]

---
## 6. BAC testing rates & laws — quality verification

These tables came in clean from ingestion (computed or hand-curated), but we verify
them here for completeness.

### `fars_bac_testing_2024`
- Computed from FARS `person.csv` — no cleaning needed, just validation
- All 51 states present, no nulls in rate columns
- Cross-check: mandatory-law states should average higher testing rates

### `bac_testing_laws`
- Hand-curated CSV — loaded directly
- All 51 states present, no nulls in classification columns
- Every state has a statute_citation entry

In [ ]:
# Cross-validate: do mandatory-law states actually test more?
validation = con.execute("""
    SELECT
        bl.mandatory_testing_law,
        COUNT(*) AS n_states,
        ROUND(AVG(bt.pct_bac_known_killed), 1) AS avg_pct_killed_tested,
        ROUND(AVG(bt.pct_bac_known_all), 1) AS avg_pct_all_tested,
        ROUND(AVG(bt.pct_bac_known_surviving), 1) AS avg_pct_surviving_tested
    FROM bac_testing_laws bl
    JOIN fars_bac_testing_2024 bt ON bl.state_fips = bt.state_fips
    GROUP BY bl.mandatory_testing_law
""").df()
print("Mandatory testing law vs actual testing rates:")
print(validation.to_string(index=False))
print()
print("Null check — bac_testing_laws:")
laws = con.execute("SELECT * FROM bac_testing_laws").df()
print(f"  mandatory_testing_law nulls: {laws['mandatory_testing_law'].isnull().sum()}")
print(f"  testing_scope nulls: {laws['testing_scope'].isnull().sum()}")
print(f"  statute_citation nulls: {laws['statute_citation'].isnull().sum()}")

---
## 7. Quality summary
All cleaned tables: 51 rows, no duplicates, minimal nulls.

In [ ]:
# Quick quality check on all cleaned tables
clean_tables = ['nhtsa_imputed_2024', 'dui_enforcement',
                'dui_criminal_status_clean', 'fars_2024_clean',
                'fars_bac_testing_2024', 'bac_testing_laws',
                'nasid_enforcement_clean', 'vehicle_impound_laws']
for t in clean_tables:
    df = con.execute(f'SELECT * FROM {t}').df()
    nulls = df.isnull().sum().sum()
    print(f'  {t}: {len(df)} rows × {len(df.columns)} cols, {nulls} total nulls')

---
## 8. DUI penalties — numeric extraction

The `dui_penalties_roadlaw` table has first-offense penalty data as free-text strings
(e.g. "$600–$2,100", "Up to 1 year", "90 days"). This section parses those into
numeric columns for analysis and visualization.

**Source:** `dui_penalties_roadlaw` (50 states) + `dui_penalties_ailawyer` (DC only)  
**Output:** `dui_penalties_numeric` — 51 rows × numeric penalty columns

### Parsing rules:
| Pattern | min | max |
|---------|-----|-----|
| `$X–$Y` / `X days–Y months` | X | Y |
| `Up to X` | 0 | X |
| `$X+` / `X days min.` | X | NULL (open-ended) |
| `None` | 0 | 0 |
| `X days` (single value) | X | X |

Time conversions: 1 month = 30 days, 1 year = 365 days, hours rounded up to ≥1 day.

In [ ]:
import re

# Load source data
roadlaw = con.execute("SELECT * FROM dui_penalties_roadlaw ORDER BY state").df()
ailawyer_dc = con.execute("""
    SELECT state, jail_1st_offense AS jail_time, fine_base AS dwi_fines,
           license_suspension AS suspension
    FROM dui_penalties_ailawyer
    WHERE state = 'District of Columbia'
""").df()

# --- Parsing functions ---

def _to_days(value, unit):
    """Convert numeric value + unit string to days."""
    unit = unit.lower().strip().rstrip('.')
    if 'hr' in unit or 'hour' in unit:
        return max(1, round(value / 24))
    elif 'day' in unit:
        return round(value)
    elif 'month' in unit:
        return round(value * 30)
    elif 'year' in unit:
        return round(value * 365)
    elif 'week' in unit:
        return round(value * 7)
    return round(value)


def parse_duration_to_days(text):
    """Parse a duration string into (min_days, max_days)."""
    if pd.isna(text) or not text:
        return (None, None)
    text = text.strip()
    
    # None / No jail
    if re.match(r'^None', text, re.IGNORECASE) or text.lower().startswith('no jail'):
        return (0, 0)
    
    # "Up to X.X units"
    m = re.match(r'Up to\s+([\d.]+)\s*(hrs?|hours?|days?|months?|years?|weeks?)', text, re.IGNORECASE)
    if m:
        return (0, _to_days(float(m.group(1)), m.group(2)))
    
    # "X units min." (mandatory minimum, no stated max)
    m = re.match(r'([\d.]+)\s*(hrs?|hours?|days?|months?)\s*min', text, re.IGNORECASE)
    if m:
        return (_to_days(float(m.group(1)), m.group(2)), None)
    
    # Range with different units: "X units–Y units"
    m = re.match(r'([\d.]+)\s*(hrs?|hours?|days?|months?|years?|weeks?)\s*[–\-—]+\s*([\d.]+)\s*(hrs?|hours?|days?|months?|years?|weeks?)', text, re.IGNORECASE)
    if m:
        return (_to_days(float(m.group(1)), m.group(2)), _to_days(float(m.group(3)), m.group(4)))
    
    # Range same unit: "X–Y units"
    m = re.match(r'([\d.]+)\s*[–\-—]\s*([\d.]+)\s*(hrs?|hours?|days?|months?|years?|weeks?)', text, re.IGNORECASE)
    if m:
        return (_to_days(float(m.group(1)), m.group(3)), _to_days(float(m.group(2)), m.group(3)))
    
    # Single value: "X units"
    m = re.match(r'([\d.]+)\s*(hrs?|hours?|days?|months?|years?|weeks?)', text, re.IGNORECASE)
    if m:
        d = _to_days(float(m.group(1)), m.group(2))
        return (d, d)
    
    return (None, None)


def parse_fine(text):
    """Parse a fine string into (min_dollars, max_dollars)."""
    if pd.isna(text) or not text:
        return (None, None)
    text = text.strip()
    amounts = [int(x.replace(',', '')) for x in re.findall(r'\$([\d,]+)', text)]
    if not amounts:
        return (None, None)
    if text.lower().startswith('up to'):
        return (0, max(amounts))
    if re.search(r'\$[\d,]+\+', text):
        return (min(amounts), None)
    if len(amounts) >= 2:
        return (min(amounts), max(amounts))
    return (amounts[0], amounts[0])


# --- Build the parsed table ---

results = []
for _, row in roadlaw.iterrows():
    jail_min, jail_max = parse_duration_to_days(row['jail_time'])
    susp_min, susp_max = parse_duration_to_days(row['suspension'])
    fine_min, fine_max = parse_fine(row['dwi_fines'])
    results.append({
        'state_name': row['state'],
        'jail_days_min': jail_min,
        'jail_days_max': jail_max,
        'suspension_days_min': susp_min,
        'suspension_days_max': susp_max,
        'fine_min_usd': fine_min,
        'fine_max_usd': fine_max,
    })

# Add DC from ailawyer
dc = ailawyer_dc.iloc[0]
jail_min, jail_max = parse_duration_to_days(dc['jail_time'])
susp_min, susp_max = parse_duration_to_days(dc['suspension'])
fine_min, fine_max = parse_fine(dc['dwi_fines'])
results.append({
    'state_name': 'District of Columbia',
    'jail_days_min': jail_min,
    'jail_days_max': jail_max,
    'suspension_days_min': susp_min,
    'suspension_days_max': susp_max,
    'fine_min_usd': fine_min,
    'fine_max_usd': fine_max,
})

penalties_numeric = pd.DataFrame(results)

# Join state_fips from states table
state_fips = con.execute("SELECT state_name, state_fips, state_abbr FROM states").df()
penalties_numeric = penalties_numeric.merge(state_fips, on='state_name', how='left')

# Reorder columns
penalties_numeric = penalties_numeric[[
    'state_fips', 'state_abbr', 'state_name',
    'jail_days_min', 'jail_days_max',
    'suspension_days_min', 'suspension_days_max',
    'fine_min_usd', 'fine_max_usd'
]]

print(f"Parsed: {len(penalties_numeric)} states")
print(f"\nSummary statistics:")
print(penalties_numeric[['jail_days_min','jail_days_max','suspension_days_min','suspension_days_max','fine_min_usd','fine_max_usd']].describe().round(0))
penalties_numeric.sort_values('state_name')

In [ ]:
# Save as parquet (interim)
penalties_numeric.to_parquet('../data/interim/dui_penalties_numeric.parquet', index=False)
print('  → data/interim/dui_penalties_numeric.parquet')

# Save to DuckDB
con.execute('DROP TABLE IF EXISTS dui_penalties_numeric')
con.execute('CREATE TABLE dui_penalties_numeric AS SELECT * FROM penalties_numeric')

# Verify
check = con.execute('SELECT COUNT(*) AS n, COUNT(jail_days_min) AS has_jail, COUNT(fine_min_usd) AS has_fine FROM dui_penalties_numeric').df()
print(f"Saved dui_penalties_numeric to DuckDB: {check['n'].iloc[0]} rows")
print(f"  jail_days_min populated: {check['has_jail'].iloc[0]}")
print(f"  fine_min_usd populated: {check['has_fine'].iloc[0]}")

---
**Next:** open `03-prepare.ipynb` to compute per-capita rates, build the master
analysis table, and package the sellable dataset.

In [ ]:
con.close()